In [ ]:
# Cell 1: Setup
!pip install -q kaggle-benchmarks datasets statsmodels pyyaml
!apt-get update -qq && apt-get install -y -qq git > /dev/null

import os, sys
from pathlib import Path

REPO_URL = "https://github.com/shqinbox/frameprobe"
REPO_NAME = "frameprobe"

if not Path(REPO_NAME).exists():
    !git clone {REPO_URL}
else:
    %cd {REPO_NAME}
    !git fetch origin
    !git reset --hard origin/main   
    %cd ..

sys.path.append(os.path.abspath(REPO_NAME))
%cd {REPO_NAME}

In [ ]:
#Cell 2: Load Experiment from YAML
from configs.experiment_config import ExperimentConfig
from pathlib import Path

config = ExperimentConfig.from_yaml("experiments/clinical_coercion_v1.yaml")

# On Kaggle, write results outside the repo clone so they appear as top-level outputs
KAGGLE_WORKING = Path("/kaggle/working")
if KAGGLE_WORKING.exists():
    config.execution.output_dir = str(KAGGLE_WORKING / "results")

print(f"Experiment: {config.name}")
print(f"Models: {config.models}")
print(f"Dataset: {config.data.source}")
print(f"Output dir: {config.execution.output_dir}")


In [ ]:
# Run the full pipeline (data -> conditions -> eval -> taxonomy -> analysis)
from benchmarks.run_experiment import run_pipeline

results_df = run_pipeline(config, skip_taxonomy=True, skip_analysis=True)
print(f"Results shape: {results_df.shape}")

In [ ]:
# (Optional) Run taxonomy classification + analysis on results
from pathlib import Path
from eval.taxonomy_classifier import BatchTaxonomyClassifier
from eval.analysis import FrameProbeAnalyzer

output_dir = Path(config.execution.output_dir)
raw_output_path = output_dir / "raw_responses.jsonl"
cache_path = output_dir / "failure_modes_cache.csv"
classified_path = output_dir / "failure_modes_final.csv"

if not classified_path.exists():
    print("Running taxonomy classification...")
    classifier = BatchTaxonomyClassifier(taxonomy_dict=config.get_taxonomy_dict())
    classifier.run(
        raw_results_path=raw_output_path,
        cache_path=cache_path,
        output_path=classified_path,
        max_workers=config.execution.max_workers,
    )
    print(f"Taxonomy saved to {classified_path}")
else:
    print(f"Using existing taxonomy file: {classified_path}")

analyzer = FrameProbeAnalyzer.from_config(config, str(classified_path))
analyzer.print_overall_performance()
analyzer.print_marginal_effects()
analyzer.print_taxonomy_breakdown()
analyzer.print_track_comparison()
analyzer.fit_interaction_model()